In [1]:
import pandas as pd, numpy as np, torch, torch.nn as nn
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix)

import sys
sys.path.insert(0, '../')
from util import *
from models.lstm_xgb import LSTM_XGB

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

SCENARIOS = [f"../CSV_Files/scene{i}/" for i in range(1)]
MODEL_NAME = "LSTM_XGB"


device: NVIDIA A30


In [2]:
def to_tensors(arr):
    X = torch.from_numpy(arr[:, 1:14].astype("float32")).unsqueeze(1)
    y = torch.from_numpy(arr[:,  -1].astype("int64"))
    return X, y


def load_scenario(scenario_dir):
    train = pd.read_csv(scenario_dir + "train.csv").to_numpy()
    val   = pd.read_csv(scenario_dir + "val.csv").to_numpy()
    test  = pd.read_csv(scenario_dir + "test.csv").to_numpy()
    X_tr_n, X_va_n, X_te_n, _, _ = preprocess_scenario(train, val, test)
    return to_tensors(X_tr_n), to_tensors(X_va_n), to_tensors(X_te_n)


def run_scenario(scenario_idx, scenario_dir, device,
                 epochs=70, lr=1e-2, weight_decay=1e-4, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(scenario_dir)

    model = LSTM_XGB(n_classes=8, lstm_hidden=32, device=device, seed=seed)
    n_params, params_by_type = count_parameters(model.backbone)  # LSTM only
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} (LSTM_XGB) ===")
        print(f"  LSTM params: {n_params:,}  breakdown: {params_by_type}")

    # ---- Stage 1: LSTM training ----
    with Timer(device) as stage1_timer:
        model.fit_lstm(X_tr, y_tr, X_val=X_va, y_val=y_va,
                       epochs=epochs, lr=lr, weight_decay=weight_decay,
                       batch_size=50, seed=seed, verbose=verbose)

    # ---- Stage 2: XGBoost fitting ----
    with Timer(device=None) as stage2_timer:   # XGBoost is CPU-bound
        model.fit_xgb(X_tr, y_tr)

    train_sec_total = stage1_timer.elapsed + stage2_timer.elapsed
    peak_mem_mb = get_gpu_peak_memory_mb(device)

    # ---- Inference timing ----
    X_te_dev = X_te.to(device)
    inf_stats = measure_inference_time(model.predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # ---- Test accuracy ----
    y_true = y_te.numpy()
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: stage1(LSTM)={stage1_timer.elapsed:.1f}s  "
              f"stage2(XGB)={stage2_timer.elapsed:.1f}s  "
              f"total={train_sec_total:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample")

    return {
        "scenario": scenario_idx, "model": "LSTM_XGB",
        "n_train": len(X_tr),
        "accuracy": acc, "precision": p, "recall": r, "f1": f,
        "confusion": cm,
        "n_params": n_params,
        "train_sec": round(train_sec_total, 2),
        "train_sec_stage1": round(stage1_timer.elapsed, 2),
        "train_sec_stage2": round(stage2_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }

In [3]:
results = []
for i, sc_dir in enumerate(SCENARIOS, start=1):
    r = run_scenario(i, sc_dir, device, epochs=70, lr=1e-2, seed=0)
    results.append(r)



=== Scenario 1 (LSTM_XGB) ===
  LSTM params: 4,744  breakdown: {'lstm': 4480, 'head': 264}


  [stage1] ep   1 | loss 0.6294 acc 0.726 | val loss 0.4017 acc 0.812


  [stage1] ep  10 | loss 0.0932 acc 0.969 | val loss 0.0933 acc 0.966


  [stage1] ep  20 | loss 0.0843 acc 0.972 | val loss 0.0760 acc 0.977


  [stage1] ep  30 | loss 0.0625 acc 0.980 | val loss 0.0489 acc 0.986


  [stage1] ep  40 | loss 0.0611 acc 0.980 | val loss 0.0566 acc 0.983


  [stage1] ep  50 | loss 0.0478 acc 0.985 | val loss 0.0438 acc 0.987


  [stage1] ep  60 | loss 0.0473 acc 0.985 | val loss 0.0441 acc 0.988


  [stage1] ep  70 | loss 0.0389 acc 0.988 | val loss 0.0373 acc 0.990


  TEST: acc=0.9869  P=0.9870  R=0.9869  F1=0.9869
  COMPUTE: stage1(LSTM)=3109.7s  stage2(XGB)=6.9s  total=3116.6s  inf=0.002ms/sample


In [4]:
summary = pd.DataFrame([
    {k: r[k] for k in ("scenario", "model", "n_train",
                       "accuracy", "precision", "recall", "f1",
                       "n_params", "train_sec", "inf_ms_per_sample",
                       "peak_mem_mb")}
    for r in results
])
cols_round = ["accuracy", "precision", "recall", "f1"]
summary[cols_round] = summary[cols_round].round(4)
print(summary.to_string(index=False))

summary.to_csv(f"{MODEL_NAME.lower()}_results_by_scenario.csv", index=False)


 scenario    model  n_train  accuracy  precision  recall     f1  n_params  train_sec  inf_ms_per_sample  peak_mem_mb
        1 LSTM_XGB   546563    0.9869      0.987  0.9869 0.9869      4744    3116.55             0.0022      19995.4


In [5]:
for r in results:
    print(f"\nScenario {r['scenario']}  ({r['model']}, "
          f"n_train={r['n_train']}, acc={r['accuracy']:.3f})")
    print(r["confusion"])



Scenario 1  (LSTM_XGB, n_train=546563, acc=0.987)
[[191   0   0   9   0   0   0   0]
 [  0 198   0   0   0   0   0   2]
 [  0   0 200   0   0   0   0   0]
 [  3   0   0 197   0   0   0   0]
 [  0   0   0   0 200   0   0   0]
 [  0   0   0   0   0 198   0   2]
 [  0   0   0   0   0   0 200   0]
 [  0   0   1   0   0   4   0 195]]
